# 飞书 OAuth Token 完整获取与持久化指南

> **文档版本**：基于飞书开放平台 OAuth v2 API(`/authen/v2/oauth/token`)
> **目标**：手把手教你获取 `user_access_token`、`refresh_token`,并实现本地持久化与自动续期
> **适用场景**：需要以「用户身份」调用飞书 OpenAPI(如创建 Wiki 知识库、访问用户文档等)

---

## 流程概览

```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│ 1. 获取授权码    │ ──→ │ 2. 换取 access_  │ ──→ │ 3. 使用 & 刷新  │
│    (code)       │     │    token        │     │  + 持久化       │
└─────────────────┘     └──────────────────┘     └─────────────────┘
                               │                        │
                               ▼                        ▼
                        ┌──────────────┐       ┌────────────────┐
                        │ refresh_token │────→│ 过期前自动续期 │
                        │ (仅能用一次)  │       │ (grant_type=   │
                        └──────────────┘       │  refresh_token)│
                                               └────────────────┘
```

### 三种 Token 的区别

| Token 类型 | 获取方式 | 有效期 | 用途 |
|---|---|---|---|
| `tenant_access_token` | app_id + app_secret | 2小时 | 应用级权限,操作企业数据 |
| `user_access_token` | OAuth 授权流程 | 约2小时 | **用户级权限**,以用户身份操作 |
| `refresh_token` | OAuth 授权时返回 | 约7天 | 用于刷新 `user_access_token`,**只能用一次** |

> ⚠️ **Wiki 知识库创建、个人文档操作等 API 必须使用 `user_access_token`**


---

## 第一步：前置准备

### 1.1 安装依赖

In [ ]:
# 只需 requests 库即可
# !pip install requests -q

import requests
import json
import time
import os
import webbrowser
import secrets
import threading
from pathlib import Path
from urllib.parse import urlencode, parse_qs, urlparse
from typing import Optional, Dict, Any

print("✅ 依赖已加载")

### 1.2 配置应用凭证

登录 [飞书开发者后台](https://open.feishu.cn/app) → 应用详情 → **凭证与基础信息** → 获取：
- **App ID** (`client_id`)
- **App Secret** (`client_secret`)

并确认已在 **安全设置 → 重定向 URL** 中添加回调地址(如 `http://localhost:8088`). 

In [ ]:
# ========== 方式一：直接填写(不推荐,容易泄露)==========
# APP_ID = "cli_xxxxxxxxxxxxxxxx"
# APP_SECRET = "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
# REDIRECT_URI = "http://localhost:8088"

# ========== 方式二：从环境变量读取(推荐)==========
# 在 .env 文件中设置：
# FEISHU_APP_ID=cli_xxx
# FEISHU_APP_SECRET=xxx

APP_ID = os.environ.get("FEISHU_APP_ID", "cli_xxxxxxxxxxxxxxxx")
APP_SECRET = os.environ.get("FEISHU_APP_SECRET", "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx")
REDIRECT_URI = os.environ.get("FEISHU_REDIRECT_URI", "http://localhost:8088")

BASE_URL = "https://open.feishu.cn/open-apis"

if "xxxxxxxx" in APP_ID or "xxxxxxxx" in APP_SECRET:
    print("⚠️ 请先替换 APP_ID 和 APP_SECRET 为真实值！")
else:
    print(f"✅ 应用配置完成 \
          APP_ID: {APP_ID[:6]}... \
          REDIRECT_URI: {REDIRECT_URI}")

### 1.3 开通 `offline_access` 权限(获取 refresh_token 必须)

> 2024年9月23日后新建的应用默认**不返回** `refresh_token`,必须手动开启. 

操作路径：
1. 飞书开放平台 → 应用详情 → **权限管理** → 搜索并开通 `offline_access`
2. 应用详情 → **安全设置** → 开启「刷新 user_access_token」开关
3. **发布应用**使配置生效

然后在授权链接的 `scope` 中拼接 `offline_access`(见下一步). 

---

## 🚀 快速通道：手动获取 Token(不用浏览器回调)

> 如果你不想/不能用代码启动本地服务器收回调,用下面任意一种方式即可：


### 方案 A：开发者后台直接复制 Token(最简单,推荐)

1. 打开 [飞书开发者后台](https://open.feishu.cn/app) → 你的应用
2. 左侧菜单找到 **API 调试台**(或 **凭证与基础信息** → 去测试)
3. 点击 **获取 user_access_token** → 扫码授权
4. 直接复制页面上显示的 `user_access_token`

拿到 token 后,直接运行下面的 cell,把它写入本地缓存：

In [ ]:
# ========== 直接填入你复制的 token ==========
MY_ACCESS_TOKEN = "u-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"  # ← 替换为你的 token
MY_REFRESH_TOKEN = None  # 可选,如果没有就不填

import json, time
from pathlib import Path

cache_path = Path.home() / f".feishu_oauth_{APP_ID[-8:]}.json"
cache = {
    "app_id": APP_ID,
    "access_token": MY_ACCESS_TOKEN,
    "refresh_token": MY_REFRESH_TOKEN,
    "expire_at": time.time() + 7200,  # 假设2小时过期
    "scope": "offline_access",
    "saved_at": time.time(),
}
cache_path.write_text(json.dumps(cache, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"✅ Token 已写入: {cache_path}")
print(f"   下次初始化 FeishuOAuthClient 时会自动加载")


### 方案 B：手动从浏览器 URL 取 code,再调 API 换 token

如果你需要 `refresh_token`(自动续期),或者后台调试台拿不到 token,按下面步骤：

**Step 1：生成授权链接,手动在浏览器打开**

In [ ]:
import requests, secrets

state = secrets.token_urlsafe(16)
scope = "offline_access"

auth_url = (
    f"https://accounts.feishu.cn/open-apis/authen/v1/authorize?"
    f"app_id={APP_ID}"
    f"&redirect_uri={requests.utils.quote(REDIRECT_URI)}"
    f"&scope={requests.utils.quote(scope)}"
    f"&state={state}"
)

print("📎 请复制下面这个链接,粘贴到浏览器地址栏打开：\n")
print(auth_url)


**Step 2：扫码授权后,浏览器地址栏会跳转到类似：**

```
http://localhost:8088?code=a61hb967bd094dge949h79bbexd16dfe&state=xxx
```

把 `code=` 后面那串字符复制下来,粘贴到下面的变量里：

In [ ]:
# ========== 粘贴你拿到的授权码 ==========
MANUAL_CODE = "a61hb967bd094dge949h79bbexd16dfe"  # ← 替换为你的 code

print(f"🔄 正在用 code 换取 token...")

resp = requests.post(
    f"{BASE_URL}/authen/v2/oauth/token",
    headers={"Content-Type": "application/json; charset=utf-8"},
    json={
        "grant_type": "authorization_code",
        "client_id": APP_ID,
        "client_secret": APP_SECRET,
        "code": MANUAL_CODE,
        "redirect_uri": REDIRECT_URI,
    },
    timeout=30,
)

data = resp.json()
print(json.dumps(data, indent=2, ensure_ascii=False))

if data.get("code", -1) == 0:
    print("\n✅ 换 token 成功！")
    print(f"   access_token: {data['access_token'][:30]}...")
    print(f"   refresh_token: {data.get('refresh_token', '无')[:30] if data.get('refresh_token') else '无'}...")
    
    # 自动保存到本地
    cache_path = Path.home() / f".feishu_oauth_{APP_ID[-8:]}.json"
    cache = {
        "app_id": APP_ID,
        "access_token": data["access_token"],
        "refresh_token": data.get("refresh_token"),
        "expire_at": time.time() + data["expires_in"],
        "refresh_expire_at": time.time() + data["refresh_token_expires_in"] if data.get("refresh_token_expires_in") else None,
        "scope": data.get("scope", ""),
        "saved_at": time.time(),
    }
    cache_path.write_text(json.dumps(cache, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"\n💾 已自动保存到: {cache_path}")
else:
    print(f"\n❌ 失败: {data.get('error_description')}")


---

## 第二步(备选)：自动获取授权码(代码启动本地服务器)

> 如果你**不想手动复制粘贴**,可以让代码自动打开浏览器、启动本地服务器收回调. 
> ⚠️ Windows 下可能因防火墙/端口问题失败,如果不行就用上面的**方案 A 或 B**. 

授权码是用户授权的临时凭证,**有效期5分钟,只能使用一次**. 

### 2.1 构造授权链接

参数说明：
| 参数 | 说明 |
|---|---|
| `client_id` | 应用 App ID |
| `redirect_uri` | 回调地址(必须与后台配置完全一致) |
| `scope` | 权限范围,多个用空格分隔;必须包含 `offline_access` 才能获得 refresh_token |
| `state` | 随机字符串,防止 CSRF 攻击 |
| `code_challenge` / `code_challenge_method` | 可选,PKCE 流程增强安全性 |

In [ ]:
# 生成随机 state(防止 CSRF)
state = secrets.token_urlsafe(16)

# 权限范围：offline_access 是必须的,其他按需添加
scope = "offline_access"  # 如需更多权限,用空格分隔,如 "offline_access bitable:app:readonly"

auth_url = (
    f"https://accounts.feishu.cn/open-apis/authen/v1/authorize?"
    f"app_id={APP_ID}"
    f"&redirect_uri={requests.utils.quote(REDIRECT_URI)}"
    f"&scope={requests.utils.quote(scope)}"
    f"&state={state}"
)

print("📎 授权链接已生成,请复制到浏览器打开并扫码授权：\n")
print(auth_url)

# 尝试自动打开浏览器
try:
    webbrowser.open(auth_url)
    print("\n🌐 已尝试自动打开浏览器")
except Exception:
    pass

### 2.2 启动本地回调服务器接收授权码

用户授权后,飞书会重定向到 `redirect_uri` 并带上 `code` 参数. 
我们启动一个临时 HTTP 服务器来接收这个回调. 

In [ ]:
from http.server import HTTPServer, BaseHTTPRequestHandler

PORT = 8088
TIMEOUT = 120  # 等待授权的最大秒数

result = {"done": False, "code": None, "error": None}

class CallbackHandler(BaseHTTPRequestHandler):
    def log_message(self, fmt, *args):
        pass  # 静默日志

    def do_GET(self):
        if self.path == "/favicon.ico":
            self.send_response(404)
            self.end_headers()
            return

        parsed = urlparse(self.path)
        query = parse_qs(parsed.query)

        code = query.get("code", [None])[0]
        returned_state = query.get("state", [None])[0]
        error = query.get("error", [None])[0]

        self.send_response(200)
        self.send_header("Content-Type", "text/html; charset=utf-8")
        self.end_headers()

        if error:
            result["error"] = f"授权失败: {error}"
            html = f"<h2 style='color:red'>❌ 授权被拒绝</h2><p>{error}</p>"
        elif code and returned_state == state:
            result["code"] = code
            html = (
                "<h2 style='color:green'>✅ 授权成功</h2>"
                "<p>授权码已获取,可以关闭此页面返回 Notebook. </p>"
            )
        else:
            result["error"] = "缺少 code 或 state 不匹配"
            html = "<h2 style='color:red'>❌ 参数错误</h2>"

        self.wfile.write(html.encode("utf-8"))
        result["done"] = True

# 启动服务器(允许端口复用)
class ReusableServer(HTTPServer):
    allow_reuse_address = True

server = ReusableServer(("127.0.0.1", PORT), CallbackHandler)
thread = threading.Thread(target=server.serve_forever, daemon=True)
thread.start()

print(f"🖥️  本地回调服务器已启动: http://127.0.0.1:{PORT}")
print(f"⏳ 等待授权中...(最多 {TIMEOUT}s)")

start = time.time()
while not result["done"] and (time.time() - start) < TIMEOUT:
    time.sleep(0.5)

server.shutdown()
server.server_close()

if result["error"]:
    print(f"❌ {result['error']}")
elif not result["done"]:
    print("❌ 授权超时,请检查是否已点击授权链接")
else:
    AUTH_CODE = result["code"]
    print(f"\n✅ 授权码获取成功！\
          code: {AUTH_CODE[:20]}...\
          长度: {len(AUTH_CODE)} 字符\
          ⚠️ 此码5分钟内有效,且只能用一次")

---

## 第三步：用授权码换取 Token(`/authen/v2/oauth/token`)

### 3.1 API 说明

| 项目 | 值 |
|---|---|
| HTTP URL | `https://open.feishu.cn/open-apis/authen/v2/oauth/token` |
| Method | `POST` |
| Content-Type | `application/json; charset=utf-8` |
| grant_type | `authorization_code` |

### 请求体示例
```json
{
    "grant_type": "authorization_code",
    "client_id": "cli_a5ca35a685b0x26e",
    "client_secret": "baBqE5um9LbFGDy3X7LcfxQX1sqpXlwy",
    "code": "a61hb967bd094dge949h79bbexd16dfe",
    "redirect_uri": "https://example.com/api/oauth/callback"
}
```

> ⚠️ **注意**：`redirect_uri` 必须与获取授权码时传入的完全一致！

In [ ]:
# 确认上一步已获取到授权码
try:
    auth_code = AUTH_CODE
except NameError:
    # 如果上一步没有运行,可以手动粘贴授权码
    auth_code = input("请粘贴授权码 code: ").strip()

print(f"🔄 正在用授权码换取 token...")

resp = requests.post(
    f"{BASE_URL}/authen/v2/oauth/token",
    headers={"Content-Type": "application/json; charset=utf-8"},
    json={
        "grant_type": "authorization_code",
        "client_id": APP_ID,
        "client_secret": APP_SECRET,
        "code": auth_code,
        "redirect_uri": REDIRECT_URI,
    },
    timeout=30
)

data = resp.json()
print(f"\n📥 原始响应：")
print(json.dumps(data, indent=2, ensure_ascii=False))

### 3.2 解析响应并保存 Token

In [ ]:
if data.get("code", -1) != 0:
    print(f"❌ 获取失败！错误码: {data.get('code')}")
    print(f"   error: {data.get('error')}")
    print(f"   error_description: {data.get('error_description')}")
    print("\n📋 常见错误排查：")
    print("   20003 - 授权码无效/已被使用")
    print("   20004 - 授权码已过期(超过5分钟)")
    print("   20071 - redirect_uri 不匹配")
    print("   20002 - client_secret 错误")
else:
    # 提取成功字段
    ACCESS_TOKEN = data["access_token"]
    REFRESH_TOKEN = data.get("refresh_token")
    EXPIRES_IN = data["expires_in"]
    REFRESH_EXPIRES_IN = data.get("refresh_token_expires_in")
    SCOPE = data.get("scope", "")
    TOKEN_TYPE = data["token_type"]

    # 计算过期时间戳
    now = time.time()
    access_expire_at = now + EXPIRES_IN
    refresh_expire_at = now + REFRESH_EXPIRES_IN if REFRESH_EXPIRES_IN else None

    print("✅ Token 获取成功！\n")
    print(f"   access_token:  {ACCESS_TOKEN[:30]}... ({len(ACCESS_TOKEN)} 字符)")
    print(f"   token_type:    {TOKEN_TYPE}")
    print(f"   expires_in:    {EXPIRES_IN}s (约 {EXPIRES_IN//3600} 小时)")
    print(f"   expire_at:     {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(access_expire_at))}")
    print(f"   scope:         {SCOPE}")
    print()
    if REFRESH_TOKEN:
        print(f"   refresh_token: {REFRESH_TOKEN[:30]}... ({len(REFRESH_TOKEN)} 字符)")
        print(f"   refresh_expires_in: {REFRESH_EXPIRES_IN}s (约 {REFRESH_EXPIRES_IN//86400} 天)")
        print(f"   refresh_expire_at:  {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(refresh_expire_at))}")
    else:
        print("   ⚠️ 未获取到 refresh_token！")
        print("      原因1：未申请 offline_access 权限")
        print("      原因2：安全设置中未开启刷新 token 开关")
        print("      原因3：scope 中未包含 offline_access")

---

## 第四步：Token 持久化(关键！)

`user_access_token` 只有约2小时有效期,`refresh_token` 只能用一次. 
**持久化是为了：下次启动程序时自动加载,无需重新扫码授权. **

### 4.1 保存到本地 JSON 文件

In [ ]:
# 持久化文件路径(放在用户主目录,安全且跨平台)
TOKEN_CACHE_PATH = Path.home() / ".feishu_oauth_tokens.json"

def save_tokens(
    access_token: str,
    refresh_token: Optional[str] = None,
    expires_in: int = 7200,
    refresh_expires_in: Optional[int] = None,
    scope: str = "",
    path: Path = TOKEN_CACHE_PATH
) -> None:
    """
    保存 token 到本地 JSON 文件,包含过期时间戳. 
    建议预留 4KB 存储空间(token 可能很长). 
    """
    now = time.time()
    cache = {
        "app_id": APP_ID,
        "access_token": access_token,
        "refresh_token": refresh_token,
        "expire_at": now + expires_in,      # access_token 过期时间戳
        "refresh_expire_at": now + refresh_expires_in if refresh_expires_in else None,
        "scope": scope,
        "saved_at": now,
    }
    path.write_text(
        json.dumps(cache, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    print(f"💾 Token 已持久化到: {path}")

# 执行保存
try:
    save_tokens(
        access_token=ACCESS_TOKEN,
        refresh_token=REFRESH_TOKEN,
        expires_in=EXPIRES_IN,
        refresh_expires_in=REFRESH_EXPIRES_IN,
        scope=SCOPE,
    )
except NameError as e:
    print(f"⚠️ 请先完成第三步获取 token: {e}")

### 4.2 从本地文件加载 Token

In [ ]:
def load_tokens(path: Path = TOKEN_CACHE_PATH) -> Optional[Dict[str, Any]]:
    """从本地加载 token,返回字典或 None"""
    if not path.exists():
        return None
    try:
        data = json.loads(path.read_text(encoding="utf-8"))
        # 校验 app_id 是否匹配(防止混用不同应用的 token)
        if data.get("app_id") != APP_ID:
            print("⚠️ 缓存的 token 属于其他应用,忽略")
            return None
        return data
    except Exception as e:
        print(f"⚠️ 加载 token 缓存失败: {e}")
        return None

# 测试加载
cached = load_tokens()
if cached:
    print("📂 已加载缓存的 token：\n")
    print(f"   access_token:  {cached['access_token'][:30]}...")
    print(f"   refresh_token: {cached['refresh_token'][:30] if cached['refresh_token'] else '无'}...")
    remaining = int(cached['expire_at'] - time.time())
    print(f"   access_token 剩余有效期: {remaining}s ({remaining//60} 分钟)")
    if cached.get("refresh_expire_at"):
        r_remaining = int(cached['refresh_expire_at'] - time.time())
        print(f"   refresh_token 剩余有效期: {r_remaining}s ({r_remaining//86400} 天)")
else:
    print("📂 没有找到本地 token 缓存")

---

## 第五步：刷新 Token(`/authen/v2/oauth/token`)

当 `user_access_token` 即将过期时,用 `refresh_token` 换取新的,**无需再次扫码**. 

> ⚠️ **refresh_token 只能用一次！** 刷新后会得到新的 refresh_token,旧的立即失效. 
> ⚠️ 刷新后请务必**更新本地保存的 refresh_token**,否则下次刷新会失败. 

### 请求参数

| 字段 | 值 |
|---|---|
| `grant_type` | `refresh_token` |
| `client_id` | 应用 App ID |
| `client_secret` | 应用 App Secret |
| `refresh_token` | 上次获取的 refresh_token |


In [ ]:
def refresh_access_token(refresh_token: str) -> Dict[str, Any]:
    """
    使用 refresh_token 刷新 user_access_token. 
    返回完整的响应数据(包含新的 access_token 和 refresh_token). 
    """
    print("🔄 正在刷新 user_access_token...")

    resp = requests.post(
        f"{BASE_URL}/authen/v2/oauth/token",
        headers={"Content-Type": "application/json; charset=utf-8"},
        json={
            "grant_type": "refresh_token",
            "client_id": APP_ID,
            "client_secret": APP_SECRET,
            "refresh_token": refresh_token,
        },
        timeout=30
    )

    result = resp.json()

    if result.get("code", -1) != 0:
        error_code = result.get("code")
        error_desc = result.get("error_description", "Unknown")

        # 常见刷新错误的排查建议
        if error_code == 20026:
            raise RuntimeError(
                f"refresh_token 无效({error_desc}). \n"
                f"可能原因：使用了旧版 v1 接口下发的 refresh_token,\n"
                f"或 token 已被使用过. 请重新走 OAuth 授权流程. "
            )
        elif error_code == 20037:
            raise RuntimeError(
                f"refresh_token 已过期({error_desc}). \n"
                f"用户授权已满365天,必须重新扫码授权. "
            )
        elif error_code == 20064 or error_code == 20073:
            raise RuntimeError(
                f"refresh_token 已被使用/撤销({error_desc}). \n"
                f"请重新走 OAuth 授权流程获取新的 refresh_token. "
            )
        elif error_code == 20074:
            raise RuntimeError(
                f"应用未开启刷新 token 权限({error_desc}). \n"
                f"请前往开发者后台 → 安全设置 → 开启刷新开关,并重新发版. "
            )
        else:
            raise RuntimeError(
                f"刷新失败: {error_desc} (错误码: {error_code})"
            )

    # 成功：提取新 token
    new_access = result["access_token"]
    new_refresh = result.get("refresh_token")
    new_expires = result["expires_in"]
    new_refresh_expires = result.get("refresh_token_expires_in")

    print("✅ 刷新成功！\n")
    print(f"   新 access_token:  {new_access[:30]}...")
    print(f"   新 refresh_token: {new_refresh[:30] if new_refresh else '无'}...")
    print(f"   新有效期: {new_expires}s")

    return result

# ========== 执行刷新 ==========
try:
    refresh_result = refresh_access_token(REFRESH_TOKEN)

    # 更新全局变量
    ACCESS_TOKEN = refresh_result["access_token"]
    REFRESH_TOKEN = refresh_result.get("refresh_token", REFRESH_TOKEN)
    EXPIRES_IN = refresh_result["expires_in"]
    REFRESH_EXPIRES_IN = refresh_result.get("refresh_token_expires_in")
    SCOPE = refresh_result.get("scope", SCOPE)

    # 立即持久化新的 token(关键！refresh_token 已更新)
    save_tokens(
        access_token=ACCESS_TOKEN,
        refresh_token=REFRESH_TOKEN,
        expires_in=EXPIRES_IN,
        refresh_expires_in=REFRESH_EXPIRES_IN,
        scope=SCOPE,
    )
    print("\n💾 新 token 已自动持久化")

except NameError:
    print("⚠️ 请先完成第三步获取初始 token")
except RuntimeError as e:
    print(f"❌ {e}")

---

## 第六步：完整封装 —— 自动续期 + 持久化

将以上所有步骤封装成一个可复用的类,实现：
1. ✅ 首次 OAuth 授权(浏览器扫码)
2. ✅ 本地 token 持久化(JSON 文件)
3. ✅ 自动判断过期并刷新
4. ✅ 自动加载上次保存的 token
5. ✅ 发送 API 请求时自动使用有效 token

In [ ]:
class FeishuOAuthClient:
    """
    飞书 OAuth 客户端(v2 API). 
    支持：授权获取 token → 本地持久化 → 自动续期 → API 调用. 
    """

    BASE_URL = "https://open.feishu.cn/open-apis"

    def __init__(
        self,
        app_id: Optional[str] = None,
        app_secret: Optional[str] = None,
        redirect_uri: str = "http://localhost:8088",
        cache_path: Optional[Path] = None,
    ):
        self.app_id = app_id or os.environ.get("FEISHU_APP_ID", "")
        self.app_secret = app_secret or os.environ.get("FEISHU_APP_SECRET", "")
        self.redirect_uri = redirect_uri

        if not self.app_id or not self.app_secret:
            raise ValueError("缺少 app_id 或 app_secret")

        # 持久化文件路径
        if cache_path is None:
            cache_key = self.app_id[-8:] if len(self.app_id) >= 8 else self.app_id
            self._cache_path = Path.home() / f".feishu_oauth_{cache_key}.json"
        else:
            self._cache_path = Path(cache_path)

        # Token 状态
        self._access_token: Optional[str] = None
        self._refresh_token: Optional[str] = None
        self._expire_at: float = 0
        self._refresh_expire_at: Optional[float] = None
        self._scope: str = ""

        # 启动时自动加载
        self._load_from_cache()

    # ─── 持久化 ───

    def _save_to_cache(self) -> None:
        """保存当前 token 到本地文件"""
        payload = {
            "app_id": self.app_id,
            "access_token": self._access_token,
            "refresh_token": self._refresh_token,
            "expire_at": self._expire_at,
            "refresh_expire_at": self._refresh_expire_at,
            "scope": self._scope,
            "saved_at": time.time(),
        }
        self._cache_path.write_text(
            json.dumps(payload, indent=2, ensure_ascii=False),
            encoding="utf-8"
        )

    def _load_from_cache(self) -> bool:
        """从本地文件加载 token,返回是否成功"""
        if not self._cache_path.exists():
            return False
        try:
            data = json.loads(self._cache_path.read_text(encoding="utf-8"))
            if data.get("app_id") != self.app_id:
                return False
            self._access_token = data.get("access_token")
            self._refresh_token = data.get("refresh_token")
            self._expire_at = data.get("expire_at", 0)
            self._refresh_expire_at = data.get("refresh_expire_at")
            self._scope = data.get("scope", "")
            print(f"📂 已从缓存加载 token ({self._cache_path.name})")
            return True
        except Exception as e:
            print(f"⚠️ 加载缓存失败: {e}")
            return False

    def clear_cache(self) -> None:
        """清除本地 token 缓存"""
        self._access_token = None
        self._refresh_token = None
        self._expire_at = 0
        if self._cache_path.exists():
            self._cache_path.unlink()
            print("🗑️  Token 缓存已清除")

    # ─── Token 管理 ───

    @property
    def access_token(self) -> Optional[str]:
        return self._access_token

    @property
    def is_expired(self) -> bool:
        """access_token 是否在60秒内过期"""
        return not self._access_token or (self._expire_at - time.time()) < 60

    @property
    def is_refresh_expired(self) -> bool:
        """refresh_token 是否已过期"""
        if not self._refresh_token or not self._refresh_expire_at:
            return True
        return self._refresh_expire_at < time.time()

    def _request_token(self, payload: Dict[str, Any]) -> Dict[str, Any]:
        """调用 /authen/v2/oauth/token 通用方法"""
        resp = requests.post(
            f"{self.BASE_URL}/authen/v2/oauth/token",
            headers={"Content-Type": "application/json; charset=utf-8"},
            json=payload,
            timeout=30,
        )
        result = resp.json()
        if result.get("code", -1) != 0:
            raise RuntimeError(
                f"Token 请求失败: {result.get('error_description')} "
                f"(code: {result.get('code')})"
            )
        return result

    def refresh(self) -> None:
        """使用 refresh_token 刷新 access_token,并自动保存"""
        if not self._refresh_token:
            raise RuntimeError("没有可用的 refresh_token,请先执行 authorize()")
        if self.is_refresh_expired:
            raise RuntimeError(
                "refresh_token 已过期,需要重新走 OAuth 授权流程. \n"
                "注意：用户授权满365天后必须重新授权. "
            )

        print("🔄 正在自动刷新 token...")
        result = self._request_token({
            "grant_type": "refresh_token",
            "client_id": self.app_id,
            "client_secret": self.app_secret,
            "refresh_token": self._refresh_token,
        })

        self._update_from_response(result)
        print("✅ Token 刷新成功并已持久化")

    def _update_from_response(self, result: Dict[str, Any]) -> None:
        """从 API 响应更新内部状态并保存"""
        now = time.time()
        self._access_token = result["access_token"]
        self._refresh_token = result.get("refresh_token", self._refresh_token)
        self._expire_at = now + result["expires_in"]
        if "refresh_token_expires_in" in result:
            self._refresh_expire_at = now + result["refresh_token_expires_in"]
        self._scope = result.get("scope", self._scope)
        self._save_to_cache()

    def ensure_token(self, auto_refresh: bool = True) -> str:
        """
        确保返回有效的 access_token. 
        如果已过期且 auto_refresh=True,自动用 refresh_token 续期. 
        """
        if self._access_token and not self.is_expired:
            return self._access_token

        if auto_refresh and self._refresh_token and not self.is_refresh_expired:
            self.refresh()
            return self._access_token

        raise RuntimeError(
            "没有有效的 access_token,且无法自动刷新. \n"
            "请执行 client.authorize() 重新获取. "
        )

    # ─── OAuth 授权流程(首次获取)───

    def authorize(
        self,
        port: int = 8088,
        timeout: int = 120,
        scope: str = "offline_access",
        auto_open_browser: bool = True,
    ) -> Dict[str, Any]:
        """
        完整的 OAuth 授权流程：打开浏览器 → 用户扫码 → 获取 token → 持久化. 
        这是获取初始 token 的唯一方式(之后可用 refresh 自动续期). 
        """
        redirect_uri = f"http://localhost:{port}"
        state = secrets.token_urlsafe(16)

        auth_url = (
            f"https://accounts.feishu.cn/open-apis/authen/v1/authorize?"
            f"app_id={self.app_id}"
            f"&redirect_uri={requests.utils.quote(redirect_uri)}"
            f"&scope={requests.utils.quote(scope)}"
            f"&state={state}"
        )

        result = {"done": False, "code": None, "error": None}
        _app_id = self.app_id
        _app_secret = self.app_secret
        _client = self

        class Handler(BaseHTTPRequestHandler):
            def log_message(self, fmt, *args):
                pass

            def do_GET(self):
                if self.path == "/favicon.ico":
                    self.send_response(404)
                    self.end_headers()
                    return
                parsed = urlparse(self.path)
                query = parse_qs(parsed.query)
                code = query.get("code", [None])[0]
                returned_state = query.get("state", [None])[0]
                error = query.get("error", [None])[0]

                self.send_response(200)
                self.send_header("Content-Type", "text/html; charset=utf-8")
                self.end_headers()

                if error:
                    result["error"] = f"授权被拒绝: {error}"
                    body = f"<h2 style='color:red'>❌ {error}</h2>"
                elif code and returned_state == state:
                    try:
                        # 用 code 换 token
                        resp = requests.post(
                            f"{_client.BASE_URL}/authen/v2/oauth/token",
                            headers={"Content-Type": "application/json; charset=utf-8"},
                            json={
                                "grant_type": "authorization_code",
                                "client_id": _app_id,
                                "client_secret": _app_secret,
                                "code": code,
                                "redirect_uri": redirect_uri,
                            },
                            timeout=30,
                        )
                        token_data = resp.json()
                        if token_data.get("code", -1) != 0:
                            result["error"] = token_data.get("error_description", "Unknown")
                            body = f"<h2 style='color:red'>❌ 换 token 失败</h2><p>{result['error']}</p>"
                        else:
                            result["code"] = code
                            _client._update_from_response(token_data)
                            body = "<h2 style='color:green'>✅ 授权成功！Token 已保存</h2>"
                    except Exception as e:
                        result["error"] = str(e)
                        body = f"<h2 style='color:red'>❌ 服务器错误</h2><p>{e}</p>"
                else:
                    result["error"] = "缺少 code 或 state 不匹配"
                    body = "<h2 style='color:red'>❌ 参数错误</h2>"

                self.wfile.write(body.encode("utf-8"))
                result["done"] = True

        class ReusableServer(HTTPServer):
            allow_reuse_address = True

        server = ReusableServer(("127.0.0.1", port), Handler)
        t = threading.Thread(target=server.serve_forever, daemon=True)
        t.start()

        print(f"\n🖥️  回调服务器: http://localhost:{port}")
        print(f"📎 授权链接:\n   {auth_url}\n")

        if auto_open_browser:
            try:
                webbrowser.open(auth_url)
                print("🌐 已自动打开浏览器")
            except Exception:
                pass

        start = time.time()
        while not result["done"] and (time.time() - start) < timeout:
            time.sleep(0.5)

        server.shutdown()
        server.server_close()

        if not result["done"]:
            raise RuntimeError(f"授权超时 ({timeout}s). 请检查是否已完成扫码. ")
        if result["error"]:
            raise RuntimeError(result["error"])

        print("\n✅ 授权完成,token 已持久化")
        return {
            "access_token": self._access_token,
            "refresh_token": self._refresh_token,
            "expire_at": self._expire_at,
        }

    # ─── API 调用 ───

    def request(
        self,
        method: str,
        path: str,
        json_data: Optional[Dict] = None,
        params: Optional[Dict] = None,
        timeout: int = 30,
        auto_refresh: bool = True,
    ) -> Dict[str, Any]:
        """
        发送带 user_access_token 的 API 请求. 
        如果 token 过期且 auto_refresh=True,自动刷新后重试一次. 
        """
        token = self.ensure_token(auto_refresh=auto_refresh)
        url = f"{self.BASE_URL}{path}"
        if params:
            url += "?" + urlencode({k: v for k, v in params.items() if v is not None})

        headers = {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json; charset=utf-8",
        }

        resp = requests.request(
            method, url, headers=headers,
            json=json_data, timeout=timeout
        )
        result = resp.json()

        # 如果因为 token 过期失败,自动刷新并重试一次
        if result.get("code") == 99991677 and auto_refresh:
            self.refresh()
            token = self.ensure_token(auto_refresh=False)
            headers["Authorization"] = f"Bearer {token}"
            resp = requests.request(
                method, url, headers=headers,
                json=json_data, timeout=timeout
            )
            result = resp.json()

        return result

    def api(self, method: str, path: str, **kwargs) -> Any:
        """发送 API 并自动检查错误码,返回 data 字段"""
        result = self.request(method, path, **kwargs)
        code = result.get("code", 0)
        if code != 0:
            raise RuntimeError(
                f"API 错误 {code}: {result.get('msg')} | path={path}")
        return result.get("data", result)

    def get_user_info(self) -> Dict[str, Any]:
        """获取当前授权用户的基本信息"""
        return self.api("GET", "/authen/v1/user_info")

print("✅ FeishuOAuthClient 类已定义")

### 6.1 使用示例：初始化 + 自动加载

In [ ]:
# 初始化客户端(会自动加载之前保存的 token)
client = FeishuOAuthClient(
    app_id=APP_ID,
    app_secret=APP_SECRET,
    redirect_uri=REDIRECT_URI,
)

print(f"\n📊 Token 状态：")
print(f"   access_token 是否存在: {bool(client.access_token)}")
print(f"   是否已过期: {client.is_expired}")
print(f"   refresh_token 是否存在: {bool(client._refresh_token)}")
print(f"   refresh_token 是否过期: {client.is_refresh_expired}")

### 6.2 如果需要首次授权(或 token 全部过期)

In [ ]:
# 只有当没有有效 token 时才需要执行
if client.is_expired and client.is_refresh_expired:
    print("🔑 需要重新授权,正在启动 OAuth 流程...\n")
    client.authorize(port=8088, scope="offline_access")
else:
    print("✅ 已有有效 token,无需重新授权")
    # 确保 token 有效(如果快过期会自动刷新)
    token = client.ensure_token()
    print(f"   当前 token: {token[:30]}...")

### 6.3 调用 API(自动处理 token)

In [ ]:
# 获取当前登录用户信息
try:
    user_info = client.get_user_info()
    print("👤 当前授权用户信息：\n")
    print(json.dumps(user_info, indent=2, ensure_ascii=False))
except RuntimeError as e:
    print(f"❌ {e}")

# 获取知识库列表(需要 user_access_token)
try:
    spaces = client.api("GET", "/wiki/v2/spaces", params={"page_size": 10})
    items = spaces.get("items", [])
    print(f"\n📚 Wiki 知识库列表(共 {len(items)} 个)：")
    for s in items[:5]:
        print(f"   - {s.get('name')} ({s.get('space_id')})")
except RuntimeError as e:
    print(f"\n❌ {e}")

---

## 附录：错误码速查表

### 获取/刷新 Token 常见错误

| 错误码 | 含义 | 排查建议 |
|---|---|---|
| `20002` | client_secret 无效 | 检查 App ID 和 App Secret 是否正确 |
| `20003` | 授权码无效 | 授权码只能用一次,是否重复使用了？ |
| `20004` | 授权码已过期 | 授权码有效期5分钟,超时需重新授权 |
| `20010` | 用户无应用使用权限 | 检查用户是否还在企业内、应用是否已启用 |
| `20024` | code/refresh_token 与 client_id 不匹配 | 不要混用不同应用的凭证 |
| `20026` | refresh_token 无效 | 可能用了旧版 v1 接口的 token,或已被使用 |
| `20037` | refresh_token 已过期 | 用户授权满365天,必须重新扫码授权 |
| `20049` | PKCE 校验失败 | code_verifier 是否正确生成和传递 |
| `20050` | 服务器内部错误 | 稍后重试 |
| `20064` / `20073` | refresh_token 已被使用/撤销 | 请重新走 OAuth 授权流程 |
| `20071` | redirect_uri 不匹配 | 确保与获取授权码时传入的完全一致 |
| `20074` | 应用未开启刷新 token 开关 | 开发者后台 → 安全设置 → 开启并重新发版 |

### API 调用常见错误

| 错误码 | 含义 | 排查建议 |
|---|---|---|
| `99991677` | token 过期 | 会自动刷新重试,如仍失败请重新授权 |
| `99991663` | 无权限 | 检查 scope 是否包含所需权限 |
| `99991661` | 非法 token | token 可能被篡改,重新获取 |


---

## 附录：持久化最佳实践

### 1. 文件位置
- **开发环境**：`~/.feishu_oauth_{app_id后缀}.json`(本 notebook 默认)
- **生产环境**：建议存入数据库或加密密钥管理服务(如 AWS Secrets Manager、阿里云 KMS)

### 2. 安全注意
- ❌ 不要把 token 提交到 Git
- ❌ 不要 hardcode 在代码里
- ✅ 文件权限设为 `600`(仅所有者可读写)
- ✅ 生产环境考虑加密存储

### 3. 多应用隔离
- 缓存文件名包含 `app_id` 后缀,防止不同应用 token 互相覆盖
- 加载时校验 `app_id` 字段,不匹配则忽略

### 4. Token 长度预留
- `access_token` 和 `refresh_token` 一般在 1~2KB,可能达到 4KB
- 数据库存储建议字段类型：`VARCHAR(8192)` 或 `TEXT`